# Seeq Application Key Rotation with Azure Key Vault

This notebook demonstrates how to:
1. Authenticate to a Seeq server
2. Retrieve an existing Seeq Application
3. Generate a new access key (rotating the credential)
4. Store the key ID and password in Azure Key Vault
5. Optionally clean up old keys

## Prerequisites
- Azure CLI logged in (`az login`)
- Permissions to read/write secrets in the target Key Vault
- Seeq credentials with permission to manage Applications

## Configuration

Update these values for your environment:

In [ ]:
# Seeq Configuration
SEEQ_SERVER_URL = "https://your-seeq-server.com"
SEEQ_USERNAME = "admin"
SEEQ_PASSWORD = "yourpassword" # Could be an env variable for better security (or from keyvault)
APPLICATION_ID = "ABC123DEF456"  # The Seeq Application ID to rotate keys for (can be retrieved by executing GET /applications from the API Docs)

# Azure Key Vault Configuration
KEYVAULT_NAME = "your-keyvault-name"
SECRET_NAME_PREFIX = "seeq-app"

# Key Rotation Configuration
KEY_EXPIRY_DAYS = 1 
KEY_LABEL = "Automated Rotation"

## Import Required Libraries

In [ ]:
import getpass
from datetime import datetime, timedelta
from typing import Optional

# Seeq SDK
from seeq import spy
from seeq.sdk.api import ApplicationsApi
from seeq.sdk.models import ApplicationKeyInputV1

# Azure SDK
from azure.identity import AzureCliCredential
from azure.keyvault.secrets import SecretClient

print("[SUCCESS] All libraries imported successfully")

## Step 1: Authenticate to Seeq

We'll use the Seeq Python SDK to authenticate. The password is prompted securely.

In [ ]:
def authenticate_to_seeq(url: str, username: str, password: Optional[str] = None) -> ApplicationsApi:
    """
    Authenticate to Seeq and return an ApplicationsApi client.
    
    Args:
        url: Seeq server URL
        username: Seeq username
        password: Seeq password (if None, will prompt)
    
    Returns:
        ApplicationsApi client instance
    """
    if password is None:
        password = getpass.getpass(f"Enter password for {username}: ")
    
    # Login to Seeq using SPy
    spy.login(url=url, username=username, password=password, quiet=True)
    
    # Create and return the Applications API client
    applications_api = ApplicationsApi(spy.client)
    
    print(f"[SUCCESS] Authenticated to Seeq server: {url}")
    return applications_api

# Authenticate
applications_api = authenticate_to_seeq(SEEQ_SERVER_URL, SEEQ_USERNAME)

## Step 2: Retrieve the Seeq Application

Get the application details to confirm it exists and view current access keys.

In [ ]:
def get_application(api: ApplicationsApi, app_id: str):
    """
    Retrieve a Seeq Application by ID.
    
    Args:
        api: ApplicationsApi client
        app_id: Seeq Application ID
    
    Returns:
        ApplicationOutputV1 object
    """
    try:
        application = api.get_application(id=app_id)
        print(f"[SUCCESS] Retrieved application: {application.name}")
        print(f"  - ID: {application.id}")
        print(f"  - Application ID: {application.application_id}")
        print(f"  - Description: {application.description or 'N/A'}")
        print(f"  - Current access keys: {len(application.access_keys)}")
        
        # Display current keys (without passwords)
        if application.access_keys:
            print("\n  Current Access Keys:")
            for key in application.access_keys:
                expiry = datetime.fromisoformat(key.expiry.replace('Z', '+00:00')) if key.expiry else None
                status = "EXPIRED" if expiry and expiry < datetime.now(expiry.tzinfo) else "ACTIVE"
                print(f"    - {key.name} (ID: {key.key_id}, Expires: {key.expiry}, Status: {status})")
        
        return application
    except Exception as e:
        print(f"[ERROR] Failed to retrieve application: {e}")
        raise

# Get the application
application = get_application(applications_api, APPLICATION_ID)

## Step 3: Generate a New Access Key

Create a new access key for the application with a specified expiration date.

In [ ]:
def generate_new_access_key(api: ApplicationsApi, app_id: str, expiry_days: int, label: str):
    """
    Generate a new access key for a Seeq Application.
    
    Args:
        api: ApplicationsApi client
        app_id: Seeq Application ID
        expiry_days: Number of days until key expires
        label: Label for the new key
    
    Returns:
        AccessKeyOutputV1 object containing key_id and password
    """
    # Calculate expiry date in ISO8601 format
    expiry_date = datetime.now(datetime.UTC) + timedelta(days=expiry_days)
    expiry_iso = expiry_date.strftime('%Y-%m-%dT%H:%M:%S.000Z')
    
    # Create the key input
    key_input = ApplicationKeyInputV1(
        expiry=expiry_iso,
        label=label
    )
    
    try:
        # Generate the new access key
        new_key = api.generate_access_key(body=key_input, id=app_id)
        
        print(f"[SUCCESS] Generated new access key")
        print(f"  - Name: {new_key.name}")
        print(f"  - Key ID: {new_key.key_id}")
        print(f"  - Expiry: {new_key.expiry}")
        print(f"  - Password: {'*' * 20} (hidden, length: {len(new_key.password)})")
        print(f"\n[WARNING] IMPORTANT: This is the only time the password will be available!")
        
        return new_key
    except Exception as e:
        print(f"[ERROR] Failed to generate access key: {e}")
        raise

# Generate a new key
timestamp = datetime.now(datetime.UTC).strftime('%Y-%m-%d %H:%M:%S UTC')
new_access_key = generate_new_access_key(
    applications_api,
    APPLICATION_ID,
    KEY_EXPIRY_DAYS,
    f"{KEY_LABEL} - {timestamp}"
)

## Step 4: Store Credentials in Azure Key Vault

Store both the access key ID and password in Azure Key Vault using Azure CLI credentials.

In [ ]:
def store_in_keyvault(keyvault_name: str, secret_prefix: str, app_id: str, 
                     key_id: str, password: str, expiry: str) -> tuple[str, str]:
    """
    Store access key credentials in Azure Key Vault.
    
    Args:
        keyvault_name: Name of the Azure Key Vault
        secret_prefix: Prefix for secret names
        app_id: Seeq Application ID
        key_id: Access key ID
        password: Access key password
        expiry: Expiry date of the key
    
    Returns:
        Tuple of (key_id_secret_name, password_secret_name)
    """
    # Create Key Vault client using Azure CLI credentials
    credential = AzureCliCredential()
    vault_url = f"https://{keyvault_name}.vault.azure.net"
    secret_client = SecretClient(vault_url=vault_url, credential=credential)
    
    # Generate secret names
    key_id_secret_name = f"{secret_prefix}-{app_id}-key-id"
    password_secret_name = f"{secret_prefix}-{app_id}-key-password"
    
    rotated_at = datetime.now(datetime.UTC).isoformat()

    try:
        # Store the key ID
        secret_client.set_secret(
            name=key_id_secret_name,
            value=key_id,
            tags={
                "seeq-application-id": app_id,
                "key-type": "access-key-id",
                "expiry": expiry,
                "rotated-at": rotated_at
            }
        )
        print(f"[SUCCESS] Stored key ID in Azure Key Vault as: {key_id_secret_name}")
        
        # Store the password
        secret_client.set_secret(
            name=password_secret_name,
            value=password,
            tags={
                "seeq-application-id": app_id,
                "key-type": "access-key-password",
                "expiry": expiry,
                "rotated-at": rotated_at
            }
        )
        print(f"[SUCCESS] Stored password in Azure Key Vault as: {password_secret_name}")
        
        return key_id_secret_name, password_secret_name
        
    except Exception as e:
        print(f"[ERROR] Failed to store secrets in Key Vault: {e}")
        raise

# Store the credentials
key_id_secret, password_secret = store_in_keyvault(
    KEYVAULT_NAME,
    SECRET_NAME_PREFIX,
    APPLICATION_ID,
    new_access_key.key_id,
    new_access_key.password,
    new_access_key.expiry
)

## Step 5: Verify Key Vault Storage (Optional)

Retrieve the secrets from Key Vault to verify they were stored correctly.

In [ ]:
def verify_keyvault_storage(keyvault_name: str, secret_names: list[str]):
    """
    Verify that secrets exist in Key Vault.
    
    Args:
        keyvault_name: Name of the Azure Key Vault
        secret_names: List of secret names to verify
    """
    credential = AzureCliCredential()
    vault_url = f"https://{keyvault_name}.vault.azure.net"
    secret_client = SecretClient(vault_url=vault_url, credential=credential)
    
    print("\nVerifying secrets in Key Vault:")
    for secret_name in secret_names:
        try:
            secret = secret_client.get_secret(secret_name)
            print(f"  [SUCCESS] {secret_name}")
            print(f"    - Version: {secret.properties.version}")
            print(f"    - Created: {secret.properties.created_on}")
            if secret.properties.tags:
                print(f"    - Tags: {secret.properties.tags}")
        except Exception as e:
            print(f"  [ERROR] {secret_name}: {e}")

# Verify storage
verify_keyvault_storage(KEYVAULT_NAME, [key_id_secret, password_secret])

## Step 6: Clean Up Old Keys (Optional)

Archive old or expired access keys to maintain security hygiene.

**WARNING**: Uncomment this section only if you want to automatically archive old keys. 
Make sure dependent services have been updated to use the new credentials first!

In [ ]:
# def cleanup_old_keys(api: ApplicationsApi, app_id: str, keep_most_recent: int = 2):
#     """
#     Archive old access keys, keeping only the most recent ones.
#     
#     Args:
#         api: ApplicationsApi client
#         app_id: Seeq Application ID
#         keep_most_recent: Number of most recent keys to keep
#     """
#     from seeq.sdk.api import AccessKeysApi
#     
#     # Get updated application info
#     application = api.get_application(id=app_id)
#     access_keys_api = AccessKeysApi(spy.client)
#     
#     # Sort keys by creation date (newest first)
#     sorted_keys = sorted(
#         application.access_keys,
#         key=lambda k: k.created_at,
#         reverse=True
#     )
#     
#     # Archive old keys
#     keys_to_archive = sorted_keys[keep_most_recent:]
#     
#     if not keys_to_archive:
#         print("No old keys to archive.")
#         return
#     
#     print(f"\nArchiving {len(keys_to_archive)} old key(s):")
#     for key in keys_to_archive:
#         try:
#             access_keys_api.archive_access_key(id=key.id)
#             print(f"  [SUCCESS] Archived key: {key.name} (ID: {key.key_id})")
#         except Exception as e:
#             print(f"  [ERROR] Failed to archive key {key.name}: {e}")
# 
# # Uncomment to enable cleanup
# # cleanup_old_keys(applications_api, APPLICATION_ID, keep_most_recent=2)

print("[WARNING] Old key cleanup is disabled by default. Uncomment the code above to enable it.")

In [ ]:
# Example: How to retrieve credentials from Key Vault in another application

def retrieve_seeq_credentials(keyvault_name: str, app_id: str, secret_prefix: str = "seeq-app"):
    """
    Retrieve Seeq Application credentials from Azure Key Vault.
    
    Returns:
        tuple: (key_id, password)
    """
    from azure.identity import DefaultAzureCredential
    from azure.keyvault.secrets import SecretClient
    
    credential = DefaultAzureCredential()
    vault_url = f"https://{keyvault_name}.vault.azure.net"
    secret_client = SecretClient(vault_url=vault_url, credential=credential)
    
    key_id = secret_client.get_secret(f"{secret_prefix}-{app_id}-key-id").value
    password = secret_client.get_secret(f"{secret_prefix}-{app_id}-key-password").value
    
    return key_id, password

# Example usage
# key_id, password = retrieve_seeq_credentials(KEYVAULT_NAME, APPLICATION_ID, SECRET_NAME_PREFIX)
# spy.login(url=SEEQ_SERVER_URL, username=key_id, password=password)

print("\n[SUCCESS] Key rotation automation complete!")
print(f"\nNew credentials are stored in Key Vault '{KEYVAULT_NAME}' as:")
print(f"  - {key_id_secret}")
print(f"  - {password_secret}")